# 03b - Compute GNN explainer attributions

Three explainers applied to the two GNN models: GNNExplainer, Integrated
Gradients, and a simplified GraphLIME. Outputs one `.npy` per
configuration.


In [ ]:
import numpy as np
import torch

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.models.gnn import GraphSAGE, TemporalGCN, get_device
from xai_blockchain_framework.utils import save_npy
from xai_blockchain_framework.xai.gnn_explainers import (
    GNNExplainerWrapper, graphlime_explain, integrated_gradients_explain,
)

set_seed()
device = get_device()
N_EVAL = 50  # smaller than ML case; GNN explainers are expensive


## Load graph tensors and trained models

In [ ]:
x_t = torch.load(PATHS.data_processed / 'elliptic_node_features.pt', weights_only=False).to(device)
edge_index = torch.load(PATHS.data_processed / 'elliptic_edge_index.pt', weights_only=False).to(device)
y_t = torch.load(PATHS.data_processed / 'elliptic_node_labels.pt', weights_only=False).to(device)
ts_t = torch.load(PATHS.data_processed / 'elliptic_time_steps.pt', weights_only=False).to(device)

tgcn = TemporalGCN(in_features=x_t.shape[1], n_timesteps=int(ts_t.max()) + 1).to(device)
tgcn.load_state_dict(torch.load(PATHS.models_dir / 'elliptic_temporalgcn.pt'))
tgcn.eval()

sage = GraphSAGE(in_features=x_t.shape[1]).to(device)
sage.load_state_dict(torch.load(PATHS.models_dir / 'elliptic_graphsage.pt'))
sage.eval()

rng = np.random.default_rng(CONFIG.random_seed)
fraud_nodes = (y_t == 1).nonzero(as_tuple=True)[0].cpu().numpy()
node_indices = rng.choice(fraud_nodes, size=min(N_EVAL, len(fraud_nodes)), replace=False)
save_npy(node_indices, PATHS.results_dir / 'gnn_eval_node_indices.npy')


## Compute attributions for each (model, explainer) pair

In [ ]:
configs = [
    ('TemporalGCN', tgcn, ts_t),
    ('GraphSAGE',   sage, None),
]
for name, model, time_steps in configs:
    tag = name.lower()

    print(f'  IG         {name}')
    ig = integrated_gradients_explain(model, x_t, edge_index, node_indices, time_steps=time_steps)
    save_npy(ig, PATHS.results_dir / f'gnn_ig_{tag}.npy')

    print(f'  GraphLIME  {name}')
    gl = graphlime_explain(model, x_t, edge_index, node_indices, time_steps=time_steps)
    save_npy(gl, PATHS.results_dir / f'gnn_graphlime_{tag}.npy')

    print(f'  GNNExplainer {name}')
    gnne = GNNExplainerWrapper(model, x_t, edge_index, time_steps=time_steps)
    ge = gnne.explain(node_indices)
    save_npy(ge, PATHS.results_dir / f'gnn_gnnexplainer_{tag}.npy')
